# BIST asistanı: LoRA eğitimi v3 (Kaggle, ücretsiz T4)

v3: eğitim verisinde kural satırı, yeni ifadeler ve daha çok nadir eylem; taban ağırlıklar Mac'te çalışan **4-bit dosyanın açılmış hâli** (Hugging Face'teki bit bit aynı kopya indirilir, sha256 denetlenir).

**Ayarlar** (sağdaki panel → *Session options*):
1. **Accelerator: GPU T4 x2** (yalnızca birincisi kullanılır).
2. **Internet: On** (4-bit model ~2,3 GB indirilir).
3. **Input**: *Add Input* → kendi `bist-v3` veri setin (`tokenized.jsonl.gz` + `train_gpu.py`).

**Çalıştırma**: sağ üstte **Save Version → Save & Run All (Commit) → Save**. Arka planda çalışır; tarayıcıyı kapatabilirsin.
İlerleme: sürüme tıkla → **Logs**. Önce kısa bir deneme koşusu (6 konuşma, ~5 dk), sonra asıl eğitim (~2,5 saat).
Bitince **Output** sekmesinden `adapter_v3.zip` dosyasını indir (~60 MB).

### GPU ve dosyalar

In [ ]:
import glob, gzip, hashlib, json, os, shutil, subprocess, sys, zipfile
import torch
assert torch.cuda.is_available(), "GPU yok: sağ panel > Session options > Accelerator > GPU T4 x2 seç."
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0), "|",
      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

IN, W, TMP = "/kaggle/input", "/kaggle/working", "/tmp/bist_unzip"
for z in glob.glob(f"{IN}/**/*.zip", recursive=True):        # zip açılmadan yüklendiyse
    zipfile.ZipFile(z).extractall(TMP)

def find(name, required=True):
    hits = glob.glob(f"{IN}/**/{name}", recursive=True) or glob.glob(f"{TMP}/**/{name}", recursive=True)
    if not hits:
        assert not required, f"{name} yok: veri setini yükleyip sağ panelde Add Input ile eklediğinden emin ol."
        return None
    if len({hashlib.md5(open(h, "rb").read()).hexdigest() for h in hits}) > 1:
        raise AssertionError(f"{name} farklı içerikle birden fazla yerde: {hits}")
    return hits[0]

shutil.copy(find("train_gpu.py"), W)
gz = find("tokenized.jsonl.gz", required=False)
if gz:
    shutil.copy(gz, W)
else:                                                       # Kaggle .gz'yi açtıysa yeniden sıkıştır
    with open(find("tokenized.jsonl"), "rb") as s, gzip.open(f"{W}/tokenized.jsonl.gz", "wb") as d:
        shutil.copyfileobj(s, d)
A = f"{W}/adapter_v3"
os.makedirs(A, exist_ok=True)
st = find("state.json", required=False)                     # yarıda kalan bir v3 koşusunun çıktısı eklendiyse sürdür
if st and not os.path.exists(f"{A}/state.json"):
    for f in ("adapter_config.json", "adapter_model.safetensors", "state.json"):
        shutil.copy(find(f), A)
    if find("train_v3_log.jsonl", required=False):
        shutil.copy(find("train_v3_log.jsonl"), W)
    print("sürdürülecek kayıt:", json.load(open(f"{A}/state.json"))["micro_step"])
n = sum(1 for _ in gzip.open(f"{W}/tokenized.jsonl.gz", "rt"))
print("konuşma:", n)

### Kütüphaneler (Mac'teki sürümlerle aynı)
Kırmızı *dependency conflicts* uyarıları normal.

In [ ]:
%pip install -q "transformers==5.24.0" "peft==0.22.0" "accelerate==1.16.0"
subprocess.run([sys.executable, "-c", "import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)"])

### 4-bit taban ağırlıklar (Mac'teki dosyanın bit bit aynısı)

In [ ]:
sys.path.insert(0, W)
import importlib, train_gpu as G
importlib.reload(G)
from huggingface_hub import hf_hub_download
MLX4 = hf_hub_download(G.MLX4_REPO, G.MLX4_FILE, revision=G.MLX4_REVISION)
h = hashlib.sha256()
with open(MLX4, "rb") as f:
    for b in iter(lambda: f.read(1 << 24), b""):
        h.update(b)
assert h.hexdigest() == G.MLX4_SHA256, "4-bit dosya Mac'tekiyle aynı değil!"
print("4-bit dosya doğrulandı:", MLX4, round(os.path.getsize(MLX4) / 1e9, 2), "GB")

### Deneme koşusu, sonra eğitim
`SDPA denemesi tamam`, sonra `loaded_from_mlx: 398` görünmeli. Her 5 adımda bir satır; `eta_min` = kalan dakika.

In [ ]:
open(f"{W}/run_train.py", "w").write('"""train_gpu.py\'yi aynen çalıştırır; yalnızca dikkat (SDPA) bellek-verimli çekirdekte kalır (v2\'deki Kaggle düzeltmesi)."""\nimport runpy, sys, torch\nimport transformers.integrations.sdpa_attention as sa\n\nsa.use_gqa_in_sdpa = lambda *a, **k: False       # GQA: enable_gqa yerine K/V tekrarı (repeat_kv)\ntorch.backends.cuda.enable_math_sdp(False)        # dikkat matrisini tümüyle kuran çekirdek kapalı\nq = torch.randn(1, 32, 8192, 128, device="cuda", dtype=torch.float16, requires_grad=True)\ntorch.nn.functional.scaled_dot_product_attention(q, q, q, is_causal=True).sum().backward()\nprint(f"SDPA denemesi tamam (8.192 token): tepe bellek {torch.cuda.max_memory_allocated() / 1e9:.2f} GB", flush=True)\ndel q\ntorch.cuda.empty_cache()\ntorch.cuda.reset_peak_memory_stats()\nsys.argv = ["train_gpu.py"] + sys.argv[1:]\nrunpy.run_path("train_gpu.py", run_name="__main__")\n')
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1", HF_HUB_DISABLE_PROGRESS_BARS="1",
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")

def run(args):
    p = subprocess.Popen([sys.executable, "run_train.py", "train", "--mlx-weights", MLX4] + args, cwd=W, env=env,
                         text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()

if not os.path.exists(f"{A}/state.json"):                   # sürdürmede deneme gerekmez
    rc = run(["--smoke", "--out", "adapter_smoke", "--log", "smoke_log.jsonl"])
    assert rc == 0, "deneme koşusu başarısız: yukarıdaki hatayı bana gönder"
    print("deneme koşusu tamam", flush=True)
rc = run(["--resume", "--out", "adapter_v3", "--log", "train_v3_log.jsonl"])
print("çıkış kodu:", rc)

### Paketle
Eğitim yarıda kalsa bile son kayıt zip'e girer (oradan sürdürülebilir).

In [ ]:
with zipfile.ZipFile(f"{W}/adapter_v3.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f"{A}/*")):
        z.write(f, "adapter_v3/" + os.path.basename(f))
    for f in ("train_v3_log.jsonl", "smoke_log.jsonl"):
        if os.path.exists(f"{W}/{f}"):
            z.write(f"{W}/{f}", f)
print("adapter_v3.zip", round(os.path.getsize(f"{W}/adapter_v3.zip") / 1e6, 1), "MB")
last = open(f"{W}/train_v3_log.jsonl").read().strip().splitlines()[-1]
print("son log satırı:", last)
assert rc == 0 and json.loads(last)["kind"] == "done", "eğitim bitmedi; zip yine de Output'ta (son kayıt dahil)"